In [35]:
from google.colab import drive
drive.mount('/content/drive')

DATA_ROOT = "/content/drive/MyDrive/CK Plus 48 8_emotions uniform"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [36]:
# ── Google Drive save paths ──
DRIVE_BASE     = "/content/drive/MyDrive/vit_emotion_training"
CHECKPOINT_DIR = f"{DRIVE_BASE}/checkpoints"
RESULTS_DIR    = f"{DRIVE_BASE}/results"

# Create the folders immediately so they're ready
import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR,    exist_ok=True)

print(f"✅ Saving everything to: {DRIVE_BASE}")
print(f"   Checkpoints → {CHECKPOINT_DIR}")
print(f"   Plots/results → {RESULTS_DIR}")

✅ Saving everything to: /content/drive/MyDrive/vit_emotion_training
   Checkpoints → /content/drive/MyDrive/vit_emotion_training/checkpoints
   Plots/results → /content/drive/MyDrive/vit_emotion_training/results


In [37]:
!pip install timm seaborn scikit-learn --quiet

In [38]:
import os
import time
import copy
import warnings
warnings.filterwarnings("ignore")

In [39]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader

In [40]:
from torchvision import datasets, transforms

In [41]:
import timm

In [42]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

In [43]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [44]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

In [45]:
LABELS = ['Anger', 'Contempt', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise']
NUM_CLASSES = len(LABELS)

In [46]:
IMAGE_SIZE  = 224

In [47]:
BATCH_SIZE  = 32

In [48]:
NUM_EPOCHS  = 30

In [49]:
LR          = 2e-4
LR_BACKBONE = 2e-5

In [50]:
WEIGHT_DECAY = 1e-4

In [51]:
PATIENCE    = 7

In [52]:
NUM_WORKERS = 2

In [53]:
SEED        = 42

In [54]:
CHECKPOINT_DIR = "checkpoints"
RESULTS_DIR    = "results"

In [55]:
def set_seed(seed: int):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

In [56]:
def get_device():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[Device] Using: {device}")
    if device.type == "cuda":
        print(f"         GPU : {torch.cuda.get_device_name(0)}")
    return device

In [57]:
def build_transforms(split: str):
    """
    Grayscale images → replicate to 3 channels → resize → normalise with
    ImageNet stats (ViT was pretrained on ImageNet RGB).
    """
    normalize = transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )

    if split == "train":
        return transforms.Compose([
            transforms.Grayscale(num_output_channels=3),   # BW → 3-ch
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.3, contrast=0.3),
            transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
            transforms.ToTensor(),
            normalize,
        ])
    else:
        return transforms.Compose([
            transforms.Grayscale(num_output_channels=3),
            transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
            transforms.ToTensor(),
            normalize,
        ])


In [58]:
def load_datasets(data_root: str):
    splits = {}
    for split in ("train", "val", "test"):
        path = os.path.join(data_root, split)
        if not os.path.isdir(path):
            # some datasets use 'validation' instead of 'val'
            alt = os.path.join(data_root, "validation" if split == "val" else split)
            if os.path.isdir(alt):
                path = alt
            else:
                print(f"[Warning] '{split}' folder not found — skipping.")
                splits[split] = None
                continue
        splits[split] = datasets.ImageFolder(path, transform=build_transforms(split))
        print(f"[Data] {split:5s}: {len(splits[split]):4d} images | "
              f"classes: {splits[split].classes}")
    return splits

In [59]:
def build_loaders(datasets_dict):
    loaders = {}
    for split, ds in datasets_dict.items():
        if ds is None:
            loaders[split] = None
            continue
        shuffle = (split == "train")
        loaders[split] = DataLoader(
            ds,
            batch_size=BATCH_SIZE,
            shuffle=shuffle,
            num_workers=NUM_WORKERS,
            pin_memory=True,
            drop_last=(split == "train"),
        )
    return loaders

In [60]:
def build_model(num_classes: int, device: torch.device) -> nn.Module:
    """
    Load pretrained ViT-B/16 from timm and replace the classifier head.
    Two parameter groups: backbone (low lr) + head (high lr).
    """
    model = timm.create_model(
        "vit_base_patch16_224",
        pretrained=True,
        num_classes=num_classes,
    )
    # The head is already replaced by timm when num_classes is passed.
    # Confirm head size:
    print(f"[Model] ViT-B/16 | head out_features={model.head.out_features}")
    model = model.to(device)
    return model

In [61]:
def get_optimizer(model: nn.Module):
    head_params     = list(model.head.parameters())
    head_ids        = {id(p) for p in head_params}
    backbone_params = [p for p in model.parameters() if id(p) not in head_ids]

    return optim.AdamW([
        {"params": backbone_params, "lr": LR_BACKBONE},
        {"params": head_params,     "lr": LR},
    ], weight_decay=WEIGHT_DECAY)


In [62]:
def run_epoch(model, loader, criterion, optimizer, device, phase="train"):
    is_train = (phase == "train")
    model.train() if is_train else model.eval()

    running_loss, running_correct, total = 0.0, 0, 0

    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)

            logits = model(imgs)
            loss   = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            preds = logits.argmax(dim=1)
            running_loss    += loss.item() * imgs.size(0)
            running_correct += (preds == labels).sum().item()
            total           += imgs.size(0)

    epoch_loss = running_loss / total
    epoch_acc  = running_correct / total
    return epoch_loss, epoch_acc

In [63]:
def train(model, loaders, device, save_dir):
    os.makedirs(save_dir, exist_ok=True)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = get_optimizer(model)
    scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS, eta_min=1e-6)

    history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
    best_val_acc   = 0.0
    best_weights   = None
    patience_count = 0

    val_loader = loaders.get("val") or loaders.get("test")
    if val_loader is None:
        raise RuntimeError("Need at least a val or test split for validation.")

    print("\n" + "═"*60)
    print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} {'Val Loss':>10} {'Val Acc':>9} {'Time':>7}")
    print("═"*60)

    for epoch in range(1, NUM_EPOCHS + 1):
        t0 = time.time()

        tr_loss, tr_acc = run_epoch(model, loaders["train"], criterion, optimizer, device, "train")
        vl_loss, vl_acc = run_epoch(model, val_loader,       criterion, optimizer, device, "val")

        scheduler.step()

        history["train_loss"].append(tr_loss)
        history["train_acc"].append(tr_acc)
        history["val_loss"].append(vl_loss)
        history["val_acc"].append(vl_acc)

        elapsed = time.time() - t0
        print(f"{epoch:>6} {tr_loss:>11.4f} {tr_acc:>10.4f} {vl_loss:>10.4f} {vl_acc:>9.4f} {elapsed:>6.1f}s")

        # Checkpoint best model
        if vl_acc > best_val_acc:
            best_val_acc = vl_acc
            best_weights = copy.deepcopy(model.state_dict())
            patience_count = 0
            torch.save({
                "epoch": epoch,
                "model_state_dict": best_weights,
                "val_acc": best_val_acc,
            }, os.path.join(save_dir, "best_vit_emotion.pth"))
            print(f"        ✓ New best val_acc={best_val_acc:.4f} — checkpoint saved.")
        else:
            patience_count += 1
            if patience_count >= PATIENCE:
                print(f"\n[Early Stop] No improvement for {PATIENCE} epochs. Stopping.")
                break

    print("═"*60)
    print(f"\n[Train] Best Val Acc = {best_val_acc:.4f}")

    # Restore best weights
    model.load_state_dict(best_weights)
    return model, history

In [64]:
def evaluate(model, loader, device, results_dir):
    os.makedirs(results_dir, exist_ok=True)
    model.eval()

    all_preds, all_labels = [], []
    with torch.no_grad():
        for imgs, labels in loader:
            imgs = imgs.to(device)
            preds = model(imgs).argmax(dim=1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())

    all_preds  = np.array(all_preds)
    all_labels = np.array(all_labels)

    # ── Accuracy ──
    acc = (all_preds == all_labels).mean()
    print(f"\n[Test] Accuracy = {acc:.4f}\n")

    # ── Full classification report ──
    report_str = classification_report(all_labels, all_preds, target_names=LABELS)
    print(report_str)

    # ── Save metrics to a text file on Drive ──
    metrics_path = os.path.join(results_dir, "metrics.txt")
    with open(metrics_path, "w") as f:
        f.write("=" * 60 + "\n")
        f.write("         EMOTION RECOGNITION — PERFORMANCE METRICS\n")
        f.write("=" * 60 + "\n\n")
        f.write(f"Overall Accuracy : {acc:.4f} ({acc*100:.2f}%)\n\n")
        f.write("Per-class metrics (Precision, Recall, F1):\n")
        f.write("-" * 60 + "\n")
        f.write(report_str)
        f.write("\n" + "=" * 60 + "\n")
    print(f"[Saved] metrics.txt → {results_dir}/")

    # ── Confusion matrix plot ──
    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt="d", xticklabels=LABELS, yticklabels=LABELS, cmap="Blues")
    plt.title("Confusion Matrix — ViT Emotion Recognition")
    plt.ylabel("True Label")
    plt.xlabel("Predicted Label")
    plt.tight_layout()
    cm_path = os.path.join(results_dir, "confusion_matrix.png")
    plt.savefig(cm_path, dpi=150)
    plt.close()
    print(f"[Saved] confusion_matrix.png → {results_dir}/")

    # ── Per-class accuracy bar chart ──
    per_class_acc = cm.diagonal() / cm.sum(axis=1)
    plt.figure(figsize=(10, 5))
    bars = plt.bar(LABELS, per_class_acc, color="steelblue", edgecolor="white")
    plt.ylim(0, 1.0)
    plt.ylabel("Accuracy")
    plt.title("Per-class Accuracy — ViT Emotion Recognition")
    plt.xticks(rotation=15)
    for bar, val in zip(bars, per_class_acc):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                 f"{val:.2f}", ha="center", va="bottom", fontsize=10)
    plt.tight_layout()
    acc_path = os.path.join(results_dir, "per_class_accuracy.png")
    plt.savefig(acc_path, dpi=150)
    plt.close()
    print(f"[Saved] per_class_accuracy.png → {results_dir}/")

In [65]:
def plot_history(history, results_dir):
    os.makedirs(results_dir, exist_ok=True)
    epochs = range(1, len(history["train_loss"]) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, history["train_loss"], label="Train Loss")
    ax1.plot(epochs, history["val_loss"],   label="Val Loss")
    ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend()

    ax2.plot(epochs, history["train_acc"], label="Train Acc")
    ax2.plot(epochs, history["val_acc"],   label="Val Acc")
    ax2.set_title("Accuracy"); ax2.set_xlabel("Epoch"); ax2.legend()

    plt.suptitle("ViT-B/16 — CK+ Emotion Recognition Training")
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "training_curves.png"), dpi=150)
    print(f"[Saved] training_curves.png → {results_dir}/")

In [66]:
set_seed(SEED)
device = get_device()

[Device] Using: cuda
         GPU : Tesla T4


In [67]:
# ── Data ──
print("\n[Data] Loading datasets …")
dataset_dict = load_datasets(DATA_ROOT)
loaders      = build_loaders(dataset_dict)

# ── Model ──
print("\n[Model] Building ViT-B/16 …")
model = build_model(NUM_CLASSES, device)

total_params = sum(p.numel() for p in model.parameters())
print(f"         Total params : {total_params:,}")

# ── Train ──
print("\n[Train] Starting training …")
model, history = train(model, loaders, device, CHECKPOINT_DIR)

# ── Plot ──
plot_history(history, RESULTS_DIR)

# ── Evaluate ──
test_loader = loaders.get("test") or loaders.get("val")
if test_loader:
    print("\n[Evaluate] Running on test set …")
    evaluate(model, test_loader, device, RESULTS_DIR)

print("\n✅ Done!")


[Data] Loading datasets …
[Data] train: 16049 images | classes: ['Anger', 'Contempt', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise']
[Data] val  : 3201 images | classes: ['Anger', 'Contempt', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise']
[Data] test : 3224 images | classes: ['Anger', 'Contempt', 'Disgust', 'Fear', 'Happiness', 'Neutral', 'Sadness', 'Surprise']

[Model] Building ViT-B/16 …
[Model] ViT-B/16 | head out_features=8
         Total params : 85,804,808

[Train] Starting training …

════════════════════════════════════════════════════════════
 Epoch  Train Loss  Train Acc   Val Loss   Val Acc    Time
════════════════════════════════════════════════════════════
     1      0.6036     0.9475     0.8710    0.8616 2089.4s
        ✓ New best val_acc=0.8616 — checkpoint saved.
     2      0.4820     0.9946     0.6835    0.9256  643.6s
        ✓ New best val_acc=0.9256 — checkpoint saved.
     3      0.4756     0.9970     0.8611    0.8638  640.4